# Machine Learning Pipeline for Binary Target - Time Series Split

A binary classification workflow that evaluates future observations using chronological train/validation/test windows and forward-only cross-validation. It compares baselines, tunes models without shuffling time, selects thresholds on validation, evaluates the later test window, and evaluates predictions on the labelled test dataset when its target is available.

**Before running:** Put `train.csv` and `test.csv` in the notebook working folder. Set `target_col`, `id_cols`, `TIME_COL`, and optional feature-engineering examples to match your columns. The labelled data must have valid timestamps in `TIME_COL`. Install the listed packages in the import cell if they are missing.

## 1. Import Libraries

Core libraries for data preparation, modeling, validation, tuning, and interpretation.

**Purpose:** Load the tools used throughout the workflow so later cells can prepare data, train models, and evaluate predictions.

In [ ]:
# Required notebook packages: %pip install pandas numpy matplotlib scikit-learn xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc, accuracy_score,
    precision_score, recall_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier


RANDOM_STATE = 42


## 2. Load and Combine the Two Datasets

The labelled and unlabelled files are concatenated immediately with an internal source tag. Row-wise feature creation then runs once on the combined table. Later, the source tag separates labelled rows for chronological train/validation/test splitting from unlabelled rows for final prediction. Imputers and encoders remain inside the model pipeline and are learned from training data only.

In [ ]:
# Configure the target column here. The file roles are detected from its presence.
target_col = "target"
train_file = pd.read_csv("train.csv")
test_file = pd.read_csv("test.csv")
if target_col in train_file.columns:
    train, test = train_file.copy(), test_file.copy()
elif target_col in test_file.columns:
    # The filenames are reversed: use the labelled file for modeling and predict the unlabelled file.
    train, test = test_file.copy(), train_file.copy()
    print("Detected reversed file roles: test.csv is labelled; train.csv is unlabelled.")
else:
    raise KeyError(f"Target column {target_col!r} must appear in one of train.csv or test.csv.")

source_col = "_dataset_source"
if source_col in train.columns or source_col in test.columns:
    raise ValueError(f"Rename the input column {source_col!r}; it is reserved by this notebook.")
train[source_col] = "labeled"
test[source_col] = "predict"
combined_data = pd.concat([train, test], ignore_index=True, sort=False)
print("Labelled modeling rows:", len(train))
print("Unlabelled prediction rows:", len(test))
print("Combined rows:", len(combined_data))
display(combined_data.head())


## 3. Quick Data Check

**Purpose:** Inspect the dataset's structure and basic quality before making assumptions about columns or preprocessing.

In [ ]:
train.info()


In [ ]:
data_check = pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique": train.nunique(dropna=False)
})

display(data_check)


In [ ]:
print("Duplicate rows:", train.duplicated().sum())
display(train.describe(include="all").T)


## 4. Target Definition

Bad debt is defined as debt payment overdue for more than three months. Label 1 represents bad debt and label 0 represents no bad debt.

**Purpose:** Confirm what the positive and negative labels mean before selecting features or comparing models.

### Label Creation from Overdue Months

This transformation applies when the released data contains an overdue-duration field measured in months and no target label is supplied.

**Purpose:** Create a binary label from an overdue-duration field only when the released data does not already provide the target.

In [ ]:
# # If the target must be derived, add it on labelled rows and sync it into combined_data.
# if "target" not in train.columns and "overdue_months" in train.columns:
#     train["target"] = (train["overdue_months"] > 3).astype(int)
#     combined_data.loc[combined_data["_dataset_source"] == "labeled", "target"] = train["target"].to_numpy()
#     print("Created target from labelled overdue_months (> 3 months overdue).")
# elif "target" in train.columns:
#     print("Using the existing target column.")
# else:
#     print("No target created. Set target_col to the correct labelled outcome column.")


## 5. Select Target and Features

**Purpose:** Separate the target from input columns and exclude identifiers or fields that should not be used for prediction.

In [ ]:
# Keep target_col configured in the data-loading cell above. Change the remaining names here.
id_cols = ["ID"]       # identifier columns retained for submission, excluded from training
drop_cols = []          # add any known leakage columns here
TIME_COL = "application_date"  # date column that defines when each prediction is made

if target_col not in train.columns:
    raise KeyError(f"Target column {target_col!r} was not found in train.csv. Update target_col above.")


In [ ]:
print(train[target_col].value_counts())
print()
print((train[target_col].value_counts(normalize=True) * 100).round(2))


In [ ]:
# Initial labelled/unlabelled feature views for diagnostics. Exclude the internal source marker.
feature_drop_cols = [col for col in [target_col, "_dataset_source"] + id_cols + drop_cols if col in train.columns]
X = train.drop(columns=feature_drop_cols).copy()
y = train[target_col].copy()
test_feature_drop_cols = [col for col in ["_dataset_source"] + id_cols + drop_cols + [target_col] if col in test.columns]
X_test = test.drop(columns=test_feature_drop_cols).copy().reindex(columns=X.columns)
print("Initial feature shapes:", X.shape, y.shape, X_test.shape)


## 6. Select Numerical and Categorical Columns

**Purpose:** Group input columns by data type so the preprocessing steps can be applied appropriately.

In [ ]:
numerical_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)


## 7. Data Quality Tests

These checks identify structural issues that can affect preprocessing, validation, and model stability.

**Purpose:** Look for unusual structure or values that may require a documented data-preparation decision.

### Missingness, Cardinality, and Constant Features

High missingness, constant variables, and very high-cardinality variables can reduce model quality or create unnecessary complexity.

**Purpose:** Find columns with extensive missing data, little variation, or many distinct values that may complicate modelling.

In [ ]:
feature_check = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_pct": (X.isna().mean() * 100).round(2),
    "unique": X.nunique(dropna=False),
    "unique_ratio": (X.nunique(dropna=False) / len(X)).round(4)
})

display(feature_check.sort_values("missing_pct", ascending=False))


In [ ]:
constant_features = feature_check[feature_check["unique"] <= 1]
high_cardinality = feature_check[feature_check["unique_ratio"] >= 0.80]

print("Constant features")
display(constant_features)

print("High-cardinality features")
display(high_cardinality)


### Infinite Values

Positive or negative infinity often appears after ratio or log transformations and should be handled before model fitting.

**Purpose:** Check for infinite numeric values that can cause preprocessing or model-fitting failures.

In [ ]:
numeric_data = X.select_dtypes(include="number")

infinite_count = pd.DataFrame({
    "positive_inf": np.isposinf(numeric_data).sum(),
    "negative_inf": np.isneginf(numeric_data).sum()
})

display(infinite_count[
    (infinite_count["positive_inf"] > 0) |
    (infinite_count["negative_inf"] > 0)
])


### Target Imbalance

A large difference between class frequencies can make accuracy misleading and may require class weighting, threshold adjustment, or resampling.

**Purpose:** Measure class proportions to understand whether the evaluation and training setup should account for unequal class counts.

In [ ]:
target_distribution = pd.DataFrame({
    "count": y.value_counts(),
    "percent": (y.value_counts(normalize=True) * 100).round(2)
})

display(target_distribution)


## 8. Outlier Tests

The IQR rule flags observations below Q1 - 1.5×IQR or above Q3 + 1.5×IQR. Outliers are diagnostic signals and are not automatically errors.

**Purpose:** Flag unusually extreme numeric observations for review; this diagnostic does not by itself justify deleting or changing them.

In [ ]:
outlier_rows = []

for col in numerical_cols:
    q1 = X[col].quantile(0.25)
    q3 = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = ((X[col] < lower) | (X[col] > upper)).sum()

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "Lower": lower,
        "Upper": upper,
        "Outliers": outlier_count,
        "Outlier_pct": outlier_count / len(X) * 100,
        "Skewness": X[col].skew()
    })

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table.sort_values("Outlier_pct", ascending=False).round(3))


## 9. Population Stability Index

PSI compares the distribution of a feature between train and test data. A common heuristic is PSI < 0.10 for small shift, 0.10–0.25 for moderate shift, and > 0.25 for substantial shift.

**Purpose:** Compare feature distributions in the labelled and unlabelled datasets to identify possible train-test shifts.

In [ ]:
def psi_numeric(train_series, test_series, bins=10):
    train_values = pd.to_numeric(train_series, errors="coerce").dropna()
    test_values = pd.to_numeric(test_series, errors="coerce").dropna()

    edges = np.unique(
        np.quantile(train_values, np.linspace(0, 1, bins + 1))
    )
    edges[0] = -np.inf
    edges[-1] = np.inf

    train_bin = pd.cut(train_values, bins=edges, include_lowest=True)
    test_bin = pd.cut(test_values, bins=edges, include_lowest=True)

    train_pct = train_bin.value_counts(
        normalize=True, sort=False
    )
    test_pct = test_bin.value_counts(
        normalize=True, sort=False
    ).reindex(train_pct.index, fill_value=0)

    train_pct = train_pct.clip(lower=1e-6)
    test_pct = test_pct.clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi


In [ ]:
def psi_categorical(train_series, test_series):
    train_pct = (
        train_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    test_pct = (
        test_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    categories = train_pct.index.union(test_pct.index)

    train_pct = train_pct.reindex(categories, fill_value=0).clip(lower=1e-6)
    test_pct = test_pct.reindex(categories, fill_value=0).clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi


In [ ]:
psi_rows = []

psi_numeric_cols = [
    col for col in numerical_cols
    if train[col].nunique(dropna=True) > 1
]

for col in psi_numeric_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Numerical",
        "PSI": psi_numeric(train[col], test[col])
    })

for col in categorical_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Categorical",
        "PSI": psi_categorical(train[col], test[col])
    })

psi_table = pd.DataFrame(psi_rows)
psi_table["Shift"] = pd.cut(
    psi_table["PSI"],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=["Small", "Moderate", "Substantial"]
)

display(psi_table.sort_values("PSI", ascending=False).round(4))


## 10. Feature Leakage Tests

Leakage occurs when a feature contains target information that would not be available at the real prediction time. Post-outcome fields such as overdue status, collections, write-offs, or recovery information are common leakage risks in credit problems.

**Purpose:** Review suspiciously predictive or post-outcome fields that might expose information unavailable at prediction time.

In [ ]:
leakage_words = [
    "target",
    "label",
    "bad_debt",
    "default",
    "overdue",
    "delinquent",
    "arrears",
    "collection",
    "writeoff",
    "write_off",
    "chargeoff",
    "charge_off",
    "recovery",
    "dpd"
]

name_leakage_candidates = [
    col for col in X.columns
    if any(word in col.lower() for word in leakage_words)
]

print("Suspicious feature names:")
print(name_leakage_candidates)


In [ ]:
numeric_with_target = train[numerical_cols + [target_col]].copy()

target_correlations = (
    numeric_with_target
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .abs()
    .sort_values(ascending=False)
)

display(target_correlations.to_frame("abs_correlation_with_target").head(20))


In [ ]:
binary_candidates = [
    col for col in X.columns
    if train[col].nunique(dropna=False) <= 2
]

copy_scores = pd.Series({
    col: (
        train[col].astype(str) ==
        train[target_col].astype(str)
    ).mean()
    for col in binary_candidates
}).sort_values(ascending=False)

display(copy_scores.to_frame("target_match_rate").head(20))


## 11. Optional Feature Engineering

**Purpose:** Create additional inputs only when the released columns and business meaning make the transformation appropriate.

### Date Features

Convert a date column into components such as year, month, quarter, and day of week.

**Purpose:** Derive calendar components from a valid date field when those components could represent useful timing patterns.

In [ ]:
# Deterministic calendar features are created once on labelled + prediction rows.
date_col = TIME_COL
if date_col in combined_data.columns:
    combined_data[date_col] = pd.to_datetime(combined_data[date_col], errors="coerce")
    combined_data["year"] = combined_data[date_col].dt.year
    combined_data["month"] = combined_data[date_col].dt.month
    combined_data["quarter"] = combined_data[date_col].dt.quarter
    combined_data["day_of_week"] = combined_data[date_col].dt.dayofweek
else:
    print(f"Skipped date features: {date_col!r} is missing from both input files.")


### Log Transformation

Useful for highly right-skewed positive numerical variables.

**Purpose:** Reduce the scale and skew of a suitable non-negative numeric variable while retaining its row-level information.

In [ ]:
# Optional deterministic log transform applied once to the joined rows.
col = "income"  # replace with a suitable non-negative numeric column
if col in combined_data.columns:
    values = pd.to_numeric(combined_data[col], errors="coerce")
    if values.dropna().ge(0).all():
        combined_data[col + "_log"] = np.log1p(values)
    else:
        print(f"Skipped log feature: {col!r} contains negative values.")
else:
    print(f"Skipped log feature: {col!r} is not present in either file.")


### Ratio Feature

Create a business ratio from two existing variables.

**Purpose:** Represent a meaningful relative quantity from two variables, taking care with zero or missing denominators.

In [ ]:
# Optional domain ratio, calculated once across the joined rows.
numerator_col = "numerator_col"
denominator_col = "denominator_col"
if numerator_col in combined_data.columns and denominator_col in combined_data.columns:
    denominator = pd.to_numeric(combined_data[denominator_col], errors="coerce").replace(0, np.nan)
    combined_data["new_ratio"] = pd.to_numeric(combined_data[numerator_col], errors="coerce") / denominator
    combined_data["new_ratio"] = combined_data["new_ratio"].replace([np.inf, -np.inf], np.nan)
else:
    print("Skipped ratio feature: replace placeholder names with real columns to enable it.")


### Interaction Feature

Represent the combined effect of two numerical variables.

**Purpose:** Represent a possible combined effect of two numeric variables for models that can use the resulting feature.

In [ ]:
# Optional interaction, calculated once across the joined rows.
col_a = "col_a"
col_b = "col_b"
if col_a in combined_data.columns and col_b in combined_data.columns:
    combined_data["interaction_feature"] = combined_data[col_a] * combined_data[col_b]
else:
    print("Skipped interaction feature: replace placeholder names with real columns to enable it.")


### Binning

Convert a continuous variable into interpretable groups.

**Purpose:** Group a continuous value into defined ranges when grouped levels are useful and the cut points are defensible.

In [ ]:
# Optional age bands created identically for both sources.
bins = [0, 25, 40, 60, np.inf]
labels = ["<=25", "26-40", "41-60", "60+"]
if "age" in combined_data.columns:
    combined_data["age_group"] = pd.cut(combined_data["age"], bins=bins, labels=labels, include_lowest=True)
else:
    print("Skipped age bands: 'age' is missing from both input files.")


### Split Train and Test (Predict)

In [ ]:
# Feature engineering above used combined_data; now separate by the source tag.
labeled_mask = combined_data["_dataset_source"].eq("labeled")
predict_mask = combined_data["_dataset_source"].eq("predict")
train = combined_data.loc[labeled_mask].drop(columns=["_dataset_source"]).copy()
test = combined_data.loc[predict_mask].drop(columns=["_dataset_source"]).copy()

if TIME_COL not in train.columns:
    raise KeyError(f"TIME_COL={TIME_COL!r} is missing from the combined data. Set it to the correct date field.")
time_values = pd.to_datetime(train[TIME_COL], errors="coerce")
if time_values.isna().any():
    raise ValueError(f"{TIME_COL!r} has missing or invalid dates in labelled data. Resolve these before splitting.")

feature_drop_cols = [c for c in [target_col, TIME_COL] + id_cols + drop_cols if c in train.columns]
X = train.drop(columns=feature_drop_cols).copy()
y = pd.to_numeric(train[target_col], errors="raise")
if y.isna().any():
    raise ValueError("The labelled target contains missing values; resolve these before training.")
if set(y.unique()) - {0, 1}:
    raise ValueError(f"The target must use binary labels 0 and 1. Found: {sorted(y.unique().tolist())}")
y = y.astype(int)
test_feature_drop_cols = [c for c in [TIME_COL] + id_cols + drop_cols + [target_col] if c in test.columns]
X_predict = test.drop(columns=test_feature_drop_cols).copy().reindex(columns=X.columns)
X = X.replace([np.inf, -np.inf], np.nan)
X_predict = X_predict.replace([np.inf, -np.inf], np.nan)
numerical_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Model inputs:", X.shape, "Unlabelled prediction inputs:", X_predict.shape)
print("Numerical features:", len(numerical_cols), "Categorical features:", len(categorical_cols))


## 12. Chronological Train / Validation / Test Split

This notebook uses a forward `TimeSeriesSplit` over unique timestamps. Rows with the same timestamp stay together; validation and test are later time windows. The `gap` parameter counts unique timestamp periods between train and validation within each fold. Set it according to any label horizon or feature lookahead that could overlap a boundary. Unlabelled rows remain separate in `X_predict`.

### Time Split Helpers

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# Keep unlabelled prediction rows separate from labelled model evaluation.
predict_drop_cols = [column for column in [target_col, TIME_COL] + id_cols + drop_cols if column in test.columns]
X_predict = test.drop(columns=predict_drop_cols).copy().reindex(columns=X.columns)
print("Unlabelled prediction rows:", len(X_predict))


def make_time_splits(X_part, time_part, n_splits=3, gap=0):
    """Forward-only CV folds; rows with the same timestamp stay together."""
    times = pd.Series(pd.to_datetime(time_part, errors="coerce"), index=X_part.index)
    if times.isna().any():
        raise ValueError("Time values contain missing or invalid timestamps.")
    unique_times = pd.Index(times.unique()).sort_values()
    splitter = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    folds = []
    for train_time_rows, valid_time_rows in splitter.split(unique_times):
        train_times = unique_times[train_time_rows]
        valid_times = unique_times[valid_time_rows]
        train_rows = np.flatnonzero(times.isin(train_times).to_numpy())
        valid_rows = np.flatnonzero(times.isin(valid_times).to_numpy())
        folds.append((train_rows, valid_rows))
    return folds


def make_time_three_way_split(X_all, y_all, time_part, n_splits=4, gap=0):
    """Use the penultimate forward fold as validation and the last as test."""
    folds = make_time_splits(X_all, time_part, n_splits=n_splits, gap=gap)
    train_rows, valid_rows = folds[-2]
    test_rows = folds[-1][1]
    return (X_all.iloc[train_rows], X_all.iloc[valid_rows], X_all.iloc[test_rows],
            y_all.iloc[train_rows], y_all.iloc[valid_rows], y_all.iloc[test_rows])


## 12. Chronological Train / Validation / Test Split

This notebook uses a forward `TimeSeriesSplit` over unique timestamps. Rows with the same timestamp stay together; validation and test are later time windows. The `gap` parameter counts unique timestamp periods between train and validation within each fold. Set it according to any label horizon or feature lookahead that could overlap a boundary. Unlabelled rows remain separate in `X_predict`.

### Create the chronological partitions

In [ ]:
X_train, X_valid, X_test_split, y_train, y_valid, y_test_split = make_time_three_way_split(
    X, y, time_values, n_splits=4, gap=0
)

print("Train dates:", time_values.loc[X_train.index].min().date(), "to", time_values.loc[X_train.index].max().date())
print("Validation dates:", time_values.loc[X_valid.index].min().date(), "to", time_values.loc[X_valid.index].max().date())
print("Test dates:", time_values.loc[X_test_split.index].min().date(), "to", time_values.loc[X_test_split.index].max().date())
print("Rows — train / valid / test / predict:", len(X_train), len(X_valid), len(X_test_split), len(X_predict))


## 13. Missing Value Methods

**Purpose:** Review alternative imputation rules and choose them within the fitted preprocessing pipeline.

### Median Imputation

Median is less sensitive to extreme values than mean.

**Purpose:** Fill numeric gaps with a robust centre that is less affected by extreme observations.

In [ ]:
numeric_median = SimpleImputer(strategy="median")


### Mean Imputation

Mean is simple and works best when extreme values are limited.

**Purpose:** Fill numeric gaps with the arithmetic average when the distribution and outlier profile make it suitable.

In [ ]:
numeric_mean = SimpleImputer(strategy="mean")


### Most Frequent Imputation

**Purpose:** Fill categorical gaps with the most common observed category.

In [ ]:
categorical_mode = SimpleImputer(strategy="most_frequent")


### Constant Category

**Purpose:** Represent missing categorical values as an explicit level when missingness itself may carry information.

In [ ]:
categorical_unknown = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)


## 14. Encoding Methods

**Purpose:** Convert categories into numeric representations that the selected estimator can process.

### One-Hot Encoding

Create one binary column for each category.

**Purpose:** Create indicator columns for categories without assigning them an artificial order.

In [ ]:
onehot = OneHotEncoder(
    handle_unknown="ignore"
)


### Ordinal Encoding

Convert categories to integer codes. The numeric order does not automatically imply business order.

**Purpose:** Map categories to integer codes; use only when the representation is appropriate because the codes imply an order to many models.

In [ ]:
ordinal = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


## 15. Scaling Methods

**Purpose:** Review transformations that put numeric features on comparable scales for scale-sensitive models.

### StandardScaler

Center variables around zero and scale them by standard deviation.

**Purpose:** Centre numeric features and scale them by their standard deviation.

In [ ]:
standard_scaler = StandardScaler()


### RobustScaler

Scale using median and interquartile range, reducing the effect of extreme values.

**Purpose:** Scale numeric features using the median and interquartile range to reduce sensitivity to extreme values.

In [ ]:
robust_scaler = RobustScaler()


## 16. Preprocessing Pipelines

**Purpose:** Combine transformations so each model receives consistently prepared inputs and transformations are fitted on training data only.

### Standard Scaling + One-Hot Encoding

**Purpose:** Use imputation, numeric scaling, and categorical indicators together for models that benefit from scaled inputs.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", ))
])

preprocess_standard = ColumnTransformer([
    ("num", numeric_pipe, numerical_cols),
    ("cat", categorical_pipe, categorical_cols)
])


### No Scaling + One-Hot Encoding

Suitable for tree-based models that do not require feature scaling.

**Purpose:** Impute missing values and encode categories without scaling numeric values, a common setup for tree models.

In [ ]:
numeric_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", ))
])

preprocess_tree = ColumnTransformer([
    ("num", numeric_pipe_tree, numerical_cols),
    ("cat", categorical_pipe_tree, categorical_cols)
])


### No Scaling + Ordinal Encoding

A compact alternative when one-hot encoding creates too many columns.

**Purpose:** Use compact category codes when their imposed numeric structure is acceptable to the chosen model.

In [ ]:
numeric_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocess_ordinal = ColumnTransformer([
    ("num", numeric_pipe_ordinal, numerical_cols),
    ("cat", categorical_pipe_ordinal, categorical_cols)
])


## 17. F1 Scoring Rule

The default criterion is binary F1 for label 1. Change `F1_AVERAGE` only if the grader specifies another averaging rule. Parameter tuning and threshold selection use this F1 rule; ROC_AUC, PR_AUC, Accuracy, Precision, and Recall are reported for comparison.

In [ ]:
# Choose the exact F1 definition used by the grader.
# "binary"   = F1 for label 1 only (the default in this notebook).
# "macro"    = calculate F1 for each class, then average them equally.
# "micro"    = aggregate class-level TP/FP/FN; for single-label data this equals accuracy.
# "weighted" = average class F1 weighted by each class row count.
F1_AVERAGE = "binary"  # change only if the grading rule requires it
F1_SCORING = {"binary": "f1", "macro": "f1_macro", "micro": "f1_micro", "weighted": "f1_weighted"}[F1_AVERAGE]
POSITIVE_LABEL = 1

def calculate_f1(y_true, y_pred):
    if F1_AVERAGE == "binary":
        return f1_score(y_true, y_pred, average="binary", pos_label=POSITIVE_LABEL, zero_division=0)
    return f1_score(y_true, y_pred, average=F1_AVERAGE, zero_division=0)

def calculate_metrics(y_true, probability, threshold=0.50):
    """Return all requested metrics; ROC_AUC / PR_AUC use probabilities."""
    y_true = pd.Series(y_true).astype(int)
    prediction = (np.asarray(probability) >= threshold).astype(int)
    has_both_classes = y_true.nunique() == 2
    if has_both_classes:
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, probability, pos_label=POSITIVE_LABEL)
        pr_auc = auc(recall_curve, precision_curve)
    else:
        pr_auc = np.nan
    return {
        "ROC_AUC": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "PR_AUC": pr_auc,
        "Accuracy": accuracy_score(y_true, prediction),
        "Precision": precision_score(y_true, prediction, pos_label=POSITIVE_LABEL, zero_division=0),
        "Recall": recall_score(y_true, prediction, pos_label=POSITIVE_LABEL, zero_division=0),
        "F1": calculate_f1(y_true, prediction),
    }

print("Tuning and threshold objective:", F1_SCORING, "| positive label:", POSITIVE_LABEL)


## 18. Fast Baseline Comparison

Fit each baseline on the earliest chronological training window. The cells report all requested metrics on train, validation, and the later labelled test holdout, using the default 0.50 cutoff. Threshold tuning comes after randomized hyperparameter search.

In [ ]:
# Shared helpers and results containers for baseline models.
baseline_rows = []
baseline_probabilities = {}
def fit_model(model_name, model, X_fit, y_fit):
    model.fit(X_fit, y_fit)
    return model
def predict_probability(model_name, model, X_eval):
    return model.predict_proba(X_eval)[:, 1]
def show_baseline_metrics(model_name, model):
    rows = []
    for split_name, X_part, y_part in [("Train", X_train, y_train), ("Validation", X_valid, y_valid), ("Test holdout", X_test_split, y_test_split)]:
        probability = predict_probability(model_name, model, X_part)
        baseline_probabilities[(model_name, split_name)] = probability
        rows.append({"Model": model_name, "Split": split_name, **calculate_metrics(y_part, probability, threshold=0.50)})
    baseline_rows.extend(rows)
    display(pd.DataFrame(rows).round(4))


### 18.1. Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("preprocess", clone(preprocess_standard)),
    ("model", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
fit_model("Logistic Regression", logistic_model, X_train, y_train)
show_baseline_metrics("Logistic Regression", logistic_model)


### 18.2. Decision Tree

In [ ]:
decision_tree_model = Pipeline([
    ("preprocess", clone(preprocess_tree)),
    ("model", DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE)),
])
fit_model("Decision Tree", decision_tree_model, X_train, y_train)
show_baseline_metrics("Decision Tree", decision_tree_model)


### 18.3. Random Forest

In [ ]:
random_forest_baseline = Pipeline([
    ("preprocess", clone(preprocess_tree)),
    ("model", RandomForestClassifier(n_estimators=120, max_depth=8, min_samples_leaf=3, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
])
fit_model("Random Forest", random_forest_baseline, X_train, y_train)
show_baseline_metrics("Random Forest", random_forest_baseline)


### 18.4. XGBoost

In [ ]:
negative_count = int((y_train != POSITIVE_LABEL).sum())
positive_count = int((y_train == POSITIVE_LABEL).sum())
scale_pos_weight = negative_count / max(positive_count, 1)
xgboost_baseline = Pipeline([
    ("preprocess", clone(preprocess_tree)),
    ("model", XGBClassifier(objective="binary:logistic", n_estimators=300, max_depth=4,
        learning_rate=0.05, min_child_weight=5, subsample=0.80, colsample_bytree=0.80,
        gamma=0.2, reg_lambda=5.0, reg_alpha=0.1, scale_pos_weight=scale_pos_weight,
        tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE)),
])
fit_model("XGBoost", xgboost_baseline, X_train, y_train)
show_baseline_metrics("XGBoost", xgboost_baseline)


### Baseline Summary

In [ ]:
baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results.round(4))


## 19. RandomizedSearchCV Cross-Validation Setup

Hyperparameter tuning uses expanding, forward-only folds from `X_train`. The validation fold in each iteration occurs after that fold’s training period.

### Forward Time-Series CV

In [ ]:
SEARCH_ITERATIONS = 10  # Reduce to 5 for a faster first pass.
SEARCH_N_JOBS = -1  # Parallelize candidate/fold fits; estimators inside search use one thread.
CV_FOLDS = 3
CV_GAP = 0  # Increase when label windows or feature lookahead overlap across boundaries.
cv_splits = make_time_splits(
    X_train, time_values.loc[X_train.index], n_splits=CV_FOLDS, gap=CV_GAP
)
print("Forward time-series CV folds:", len(cv_splits))


## 20. RandomizedSearchCV: Random Forest

Run this cell to tune the Random Forest. Depth, leaf size, and tree count stay bounded to control runtime and overfitting.

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocess", clone(preprocess_tree)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    ]),
    param_distributions={
        "model__n_estimators": [100, 150, 200, 250, 300],
        "model__max_depth": [4, 6, 8, 10, 12, 14, None],
        "model__min_samples_leaf": [2, 3, 5, 7, 10],
        "model__max_features": [0.4, 0.6, 0.8, 1.0],
        "model__class_weight": ["balanced", "balanced_subsample", None],
    },
    n_iter=SEARCH_ITERATIONS, scoring=F1_SCORING, cv=cv_splits,
    random_state=RANDOM_STATE, n_jobs=SEARCH_N_JOBS, verbose=1,
    refit=True, error_score="raise", return_train_score=False,
)
rf_search.fit(X_train, y_train)
rf_best_params = {key.removeprefix("model__"): value for key, value in rf_search.best_params_.items()}
print("Best RF CV F1:", round(rf_search.best_score_, 4))
print("Best RF parameters:", rf_best_params)


## 21. RandomizedSearchCV: XGBoost

Run this cell to tune XGBoost independently. The ranges bound tree depth and estimator count; `n_jobs=-1` uses available CPU cores.

In [ ]:
negative_count = int((y_train != POSITIVE_LABEL).sum())
positive_count = int((y_train == POSITIVE_LABEL).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

xgb_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocess", clone(preprocess_tree)),
        ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1)),
    ]),
    param_distributions={
        "model__n_estimators": [100, 150, 200, 250, 300],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.02, 0.05, 0.08, 0.10, 0.15],
        "model__min_child_weight": [1, 3, 5, 7, 10],
        "model__subsample": [0.70, 0.80, 0.90, 1.0],
        "model__colsample_bytree": [0.70, 0.80, 0.90, 1.0],
        "model__reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
        "model__scale_pos_weight": [1.0, scale_pos_weight],
    },
    n_iter=SEARCH_ITERATIONS, scoring=F1_SCORING, cv=cv_splits,
    random_state=RANDOM_STATE, n_jobs=SEARCH_N_JOBS, verbose=1,
    refit=True, error_score="raise", return_train_score=False,
)
xgb_search.fit(X_train, y_train)
xgb_best_params = {key.removeprefix("model__"): value for key, value in xgb_search.best_params_.items()}
print("Best XGBoost CV F1:", round(xgb_search.best_score_, 4))
print("Best XGBoost parameters:", xgb_best_params)


## 22. RandomizedSearchCV: Logistic Regression

This tunes Logistic Regression separately from the tree models. RandomizedSearchCV chooses regularization strength `C`, penalty, and class weighting to maximize the selected F1 on the same CV folds. `liblinear` works with the sparse one-hot encoded features.

In [ ]:
logistic_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocess", clone(preprocess_standard)),
        ("model", LogisticRegression(solver="liblinear", max_iter=1500, random_state=RANDOM_STATE)),
    ]),
    param_distributions={
        "model__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "model__penalty": ["l1", "l2"],
        "model__class_weight": [None, "balanced"],
    },
    n_iter=SEARCH_ITERATIONS, scoring=F1_SCORING, cv=cv_splits,
    random_state=RANDOM_STATE, n_jobs=SEARCH_N_JOBS, verbose=1,
    refit=True, error_score="raise", return_train_score=False,
)
logistic_search.fit(X_train, y_train)
logistic_best_params = {key.removeprefix("model__"): value for key, value in logistic_search.best_params_.items()}
print("Best Logistic Regression CV F1:", round(logistic_search.best_score_, 4))
print("Best Logistic Regression parameters:", logistic_best_params)


### 22.1 Optional Optuna Alternative (commented out)

To use Optuna instead, skip/comment out the three RandomizedSearchCV `.fit(...)` cells above, install Optuna if needed, then uncomment and run the entire code block below. It reuses the same forward-only `cv_splits`, optimizes the notebook F1 rule at threshold 0.50, and assigns the same `*_best_params` variables used downstream.


In [ ]:
# # Optional alternative to RandomizedSearchCV. Uncomment this entire block to use it.
# import optuna
# from sklearn.base import clone
#
# OPTUNA_TRIALS = 20
# OPTUNA_SEED = RANDOM_STATE
#
# def _optuna_mean_cv_f1(make_model):
#     fold_scores = []
#     for train_idx, valid_idx in cv_splits:  # Forward-only folds from make_time_splits.
#         fold_model = make_model()
#         fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
#         fold_probability = fold_model.predict_proba(X_train.iloc[valid_idx])[:, 1]
#         fold_prediction = (fold_probability >= 0.50).astype(int)
#         fold_scores.append(calculate_f1(y_train.iloc[valid_idx], fold_prediction))
#     return float(np.mean(fold_scores))
#
# def _rf_objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
#         "max_depth": trial.suggest_categorical("max_depth", [4, 6, 8, 10, 12, None]),
#         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 10),
#         "max_features": trial.suggest_float("max_features", 0.4, 1.0),
#         "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
#     }
#     def make_model():
#         return Pipeline([("preprocess", clone(preprocess_tree)),
#                         ("model", RandomForestClassifier(**params, n_jobs=1, random_state=RANDOM_STATE))])
#     return _optuna_mean_cv_f1(make_model)
# rf_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED))
# rf_study.optimize(_rf_objective, n_trials=OPTUNA_TRIALS, n_jobs=1)
# rf_best_params = rf_study.best_params
# print("Optuna RF CV F1:", rf_study.best_value, "| params:", rf_best_params)
#
# def _xgb_objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
#         "max_depth": trial.suggest_int("max_depth", 3, 8),
#         "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
#         "subsample": trial.suggest_float("subsample", 0.70, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 1.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
#         "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", list(dict.fromkeys([1.0, float(scale_pos_weight)]))),
#     }
#     def make_model():
#         return Pipeline([("preprocess", clone(preprocess_tree)),
#                         ("model", XGBClassifier(**params, eval_metric="logloss", n_jobs=1, random_state=RANDOM_STATE))])
#     return _optuna_mean_cv_f1(make_model)
# xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED))
# xgb_study.optimize(_xgb_objective, n_trials=OPTUNA_TRIALS, n_jobs=1)
# xgb_best_params = xgb_study.best_params
# print("Optuna XGBoost CV F1:", xgb_study.best_value, "| params:", xgb_best_params)
#
# def _logistic_objective(trial):
#     params = {
#         "C": trial.suggest_float("C", 0.001, 100.0, log=True),
#         "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
#         "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
#     }
#     def make_model():
#         return Pipeline([("preprocess", clone(preprocess_standard)),
#                         ("model", LogisticRegression(**params, solver="liblinear", max_iter=1500, random_state=RANDOM_STATE))])
#     return _optuna_mean_cv_f1(make_model)
# logistic_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED))
# logistic_study.optimize(_logistic_objective, n_trials=OPTUNA_TRIALS, n_jobs=1)
# logistic_best_params = logistic_study.best_params
# print("Optuna Logistic Regression CV F1:", logistic_study.best_value, "| params:", logistic_best_params)


## 23. Fit the Three Tuned Models on Train

Use the best parameters from the three separate RandomizedSearchCV searches. These fitted models support validation threshold selection and the first test-holdout evaluation.

In [ ]:
tuned_models = {
    "Logistic Regression": Pipeline([("preprocess", clone(preprocess_standard)), ("model", LogisticRegression(**logistic_best_params, solver="liblinear", max_iter=1500, random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("preprocess", clone(preprocess_tree)), ("model", RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("preprocess", clone(preprocess_tree)), ("model", XGBClassifier(**xgb_best_params, eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE))]),
}
for model_name, model in tuned_models.items():
    fit_model(model_name, model, X_train, y_train)

def find_best_f1_threshold(y_true, probability):
    # Search a dense set of cutoffs from validation scores, including all-positive/all-negative boundaries.
    probability = np.asarray(probability)
    candidates = np.unique(np.r_[0.0, np.quantile(probability, np.linspace(0.01, 0.99, 199)), 1.0])
    scores = [calculate_f1(y_true, (probability >= cutoff).astype(int)) for cutoff in candidates]
    best_index = int(np.argmax(scores))
    return float(candidates[best_index]), float(scores[best_index])

tuned_probabilities = {}
for model_name, model in tuned_models.items():
    tuned_probabilities[(model_name, "Train")] = predict_probability(model_name, model, X_train)
    tuned_probabilities[(model_name, "Validation")] = predict_probability(model_name, model, X_valid)
    tuned_probabilities[(model_name, "Test holdout") ] = predict_probability(model_name, model, X_test_split)


## 24. Threshold Tuning: RF

This model gets its own probability threshold, selected only on the validation period by F1 for label 1. The resulting threshold is used unchanged for train/validation/test reporting.

In [ ]:
rf_threshold, rf_valid_f1 = find_best_f1_threshold(y_valid, tuned_probabilities[("Random Forest", "Validation")])
thresholds = globals().get("thresholds", {})
thresholds["Random Forest"] = rf_threshold
print("Random Forest best validation threshold:", rf_threshold)
print("Random Forest validation F1 at threshold:", round(rf_valid_f1, 4))
rows = []
for split_name, y_part in [("Train", y_train), ("Validation", y_valid), ("Test holdout", y_test_split)]:
    rows.append({"Model": "Random Forest", "Split": split_name, **calculate_metrics(y_part, tuned_probabilities[("Random Forest", split_name)], threshold=rf_threshold)})
display(pd.DataFrame(rows).round(4))


## 25. Threshold Tuning: XGBoost

This model gets its own probability threshold, selected only on the validation period by F1 for label 1. The resulting threshold is used unchanged for train/validation/test reporting.

In [ ]:
xgboost_threshold, xgboost_valid_f1 = find_best_f1_threshold(y_valid, tuned_probabilities[("XGBoost", "Validation")])
thresholds = globals().get("thresholds", {})
thresholds["XGBoost"] = xgboost_threshold
print("XGBoost best validation threshold:", xgboost_threshold)
print("XGBoost validation F1 at threshold:", round(xgboost_valid_f1, 4))
rows = []
for split_name, y_part in [("Train", y_train), ("Validation", y_valid), ("Test holdout", y_test_split)]:
    rows.append({"Model": "XGBoost", "Split": split_name, **calculate_metrics(y_part, tuned_probabilities[("XGBoost", split_name)], threshold=xgboost_threshold)})
display(pd.DataFrame(rows).round(4))


## 26. Threshold Tuning: Logistic Regression

Choose this validation threshold separately for Logistic Regression. The same threshold is used for train/validation/test metrics.

In [ ]:
logistic_threshold, logistic_valid_f1 = find_best_f1_threshold(
    y_valid, tuned_probabilities[("Logistic Regression", "Validation")]
)
thresholds["Logistic Regression"] = logistic_threshold
print("Logistic Regression validation threshold:", logistic_threshold)
print("Validation F1:", round(logistic_valid_f1, 4))

logistic_threshold_rows = []
for split_name, y_part in [("Train", y_train), ("Validation", y_valid), ("Test holdout", y_test_split)]:
    logistic_threshold_rows.append({
        "Model": "Logistic Regression",
        "Split": split_name,
        **calculate_metrics(y_part, tuned_probabilities[("Logistic Regression", split_name)], logistic_threshold),
    })
display(pd.DataFrame(logistic_threshold_rows).round(4))


## 27. Ensemble Experiment: Easy Probability Blends

Ensemble means: average the member models' predicted probabilities for label 1, then choose one threshold for that average using validation F1. For example, `(RF + XGBoost probabilities) / 2`. The ensemble gets its own threshold; do not average the component thresholds.

Run the helper once, then run one or both examples. The first compares trees; the second shows whether adding the linear model helps.

### 27.1 Blend helper (run once before either experiment)

This small helper averages the probability arrays returned by the selected tuned models. Run it once, then run one or both blend examples below.

In [ ]:
def average_model_probabilities(model_names, X_part):
    """Average label-1 probabilities from the named tuned models."""
    probabilities = [predict_probability(name, tuned_models[name], X_part) for name in model_names]
    return np.mean(probabilities, axis=0)

ensemble_members = {}
ensemble_thresholds = {}
ensemble_probabilities = {}


### 27.2 Tree-only ensemble: RF + XGBoost

In [ ]:
tree_model_names = ["Random Forest", "XGBoost"]
tree_blend_probability = {
    "Train": average_model_probabilities(tree_model_names, X_train),
    "Validation": average_model_probabilities(tree_model_names, X_valid),
    "Test holdout": average_model_probabilities(tree_model_names, X_test_split),
}

tree_blend_threshold, tree_blend_valid_f1 = find_best_f1_threshold(
    y_valid, tree_blend_probability["Validation"]
)
print("Validation threshold:", round(tree_blend_threshold, 4))
print("Validation F1:", round(tree_blend_valid_f1, 4))

ensemble_members["RF + XGBoost"] = tree_model_names
ensemble_thresholds["RF + XGBoost"] = tree_blend_threshold
for split_name, probability in tree_blend_probability.items():
    ensemble_probabilities[("RF + XGBoost", split_name)] = probability

tree_blend_results = pd.DataFrame([
    {"Model": "RF + XGBoost", "Split": split_name,
     **calculate_metrics(y_part, tree_blend_probability[split_name], tree_blend_threshold)}
    for split_name, y_part in [("Train", y_train), ("Validation", y_valid), ("Test holdout", y_test_split)]
])
display(tree_blend_results.round(4))


### 27.3 Add Logistic Regression to the ensemble

In [ ]:
linear_tree_model_names = ["Logistic Regression", "Random Forest", "XGBoost"]
linear_tree_blend_probability = {
    "Train": average_model_probabilities(linear_tree_model_names, X_train),
    "Validation": average_model_probabilities(linear_tree_model_names, X_valid),
    "Test holdout": average_model_probabilities(linear_tree_model_names, X_test_split),
}

linear_tree_threshold, linear_tree_valid_f1 = find_best_f1_threshold(
    y_valid, linear_tree_blend_probability["Validation"]
)
print("Validation threshold:", round(linear_tree_threshold, 4))
print("Validation F1:", round(linear_tree_valid_f1, 4))

linear_tree_name = "Logistic + RF + XGBoost"
ensemble_members[linear_tree_name] = linear_tree_model_names
ensemble_thresholds[linear_tree_name] = linear_tree_threshold
for split_name, probability in linear_tree_blend_probability.items():
    ensemble_probabilities[(linear_tree_name, split_name)] = probability

linear_tree_results = pd.DataFrame([
    {"Model": linear_tree_name, "Split": split_name,
     **calculate_metrics(y_part, linear_tree_blend_probability[split_name], linear_tree_threshold)}
    for split_name, y_part in [("Train", y_train), ("Validation", y_valid), ("Test holdout", y_test_split)]
])
display(linear_tree_results.round(4))


## 28. Compare Tuned Models and Ensembles

Use validation F1 to choose a candidate workflow. Test-holdout metrics are reported for an honest later-period comparison, but should not be used to tune parameters or thresholds.

In [ ]:
validation_comparison = pd.concat([
    pd.DataFrame([
        {"Model": name, "Threshold": thresholds[name], **calculate_metrics(y_valid, tuned_probabilities[(name, "Validation")], thresholds[name])}
        for name in tuned_models
    ]),
    pd.DataFrame([
        {"Model": name, "Threshold": ensemble_thresholds[name], **calculate_metrics(y_valid, ensemble_probabilities[(name, "Validation")], ensemble_thresholds[name])}
        for name in ensemble_members
    ])
], ignore_index=True).sort_values("F1", ascending=False).reset_index(drop=True)
display(validation_comparison.round(4))
print("Select using validation F1 and practical constraints. Test results below are final diagnostics, not tuning feedback.")


## 29. Refit on Train + Validation and Re-evaluate the Same Test Holdout

First report each tuned model on test after fitting only on train. Then refit it on train + validation and evaluate again on that same later test window. Keep the validation-selected threshold fixed so the score change reflects refitting with more labelled rows. The validation period becomes part of training in this second fit, so it is no longer an independent evaluation split. The test window remains excluded from every fit and threshold search.

In [ ]:
X_development = pd.concat([X_train, X_valid]).sort_index()
y_development = pd.concat([y_train, y_valid]).sort_index()

refit_holdout_rows = []
refit_development_rows = []
models_train_valid = {}
for model_name in tuned_models:
    if model_name == "Logistic Regression":
        model = Pipeline([("preprocess", clone(preprocess_standard)), ("model", LogisticRegression(**logistic_best_params, solver="liblinear", max_iter=1500, random_state=RANDOM_STATE))])
    elif model_name == "Random Forest":
        model = Pipeline([("preprocess", clone(preprocess_tree)), ("model", RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=RANDOM_STATE))])
    elif model_name == "XGBoost":
        model = Pipeline([("preprocess", clone(preprocess_tree)), ("model", XGBClassifier(**xgb_best_params, eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE))])
    fit_model(model_name, model, X_development, y_development)
    models_train_valid[model_name] = model
    probability_dev = predict_probability(model_name, model, X_development)
    probability_holdout = predict_probability(model_name, model, X_test_split)
    refit_development_rows.append({"Model": model_name, "Split": "Train + Validation (in-sample)", **calculate_metrics(y_development, probability_dev, thresholds[model_name])})
    refit_holdout_rows.append({"Model": model_name, "Split": "Test holdout after Train + Validation refit", **calculate_metrics(y_test_split, probability_holdout, thresholds[model_name])})

pre_refit_test = pd.DataFrame([
    {"Model": name, "Split": "Test holdout before refit (train only)", **calculate_metrics(y_test_split, tuned_probabilities[(name, "Test holdout")], thresholds[name])}
    for name in tuned_models
])
refit_comparison = pd.concat([pre_refit_test, pd.DataFrame(refit_development_rows), pd.DataFrame(refit_holdout_rows)], ignore_index=True)
display(refit_comparison.round(4))


## 30. Holdout ROC AUC and PR AUC Curves

Compare every baseline model, every tuned model, and the selected final ensemble on the chronological test holdout. The ensemble uses tuned members trained before the holdout. The later full-data refit is for prediction and must not be evaluated against this holdout.


In [ ]:
# Select the final ensemble from validation results before comparing holdout curves.
# Change to "Logistic + RF + XGBoost" only after reviewing validation_comparison.
FINAL_ENSEMBLE_NAME = "RF + XGBoost"
if FINAL_ENSEMBLE_NAME not in ensemble_members:
    raise ValueError(f"Unknown final ensemble: {FINAL_ENSEMBLE_NAME!r}")
curve_probabilities = {}
for model_name in ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]:
    curve_probabilities[f"{model_name} (baseline)"] = baseline_probabilities[(model_name, "Test holdout")] 
for model_name in tuned_models:
    curve_probabilities[f"{model_name} (tuned)"] = tuned_probabilities[(model_name, "Test holdout")] 
curve_probabilities[f"{FINAL_ENSEMBLE_NAME} (final ensemble)"] = ensemble_probabilities[(FINAL_ENSEMBLE_NAME, "Test holdout")] 
y_curve = y_test_split

# Draw one figure per model so its ROC and PR curves can be inspected separately.
for model_label, probability in curve_probabilities.items():
    fpr, tpr, _ = roc_curve(y_curve, probability, pos_label=POSITIVE_LABEL)
    roc_auc = roc_auc_score(y_curve, probability)
    precision, recall, _ = precision_recall_curve(y_curve, probability, pos_label=POSITIVE_LABEL)
    pr_auc = auc(recall, precision)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(fpr, tpr, color="tab:blue", label=f"ROC AUC = {roc_auc:.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Chance")
    axes[0].set(title=f"ROC curve — {model_label}", xlabel="False Positive Rate", ylabel="True Positive Rate")
    axes[0].legend(loc="best")

    axes[1].plot(recall, precision, color="tab:orange", label=f"PR AUC = {pr_auc:.3f}")
    axes[1].axhline(float(np.mean(y_curve == POSITIVE_LABEL)), linestyle="--", color="gray", linewidth=1, label="Positive prevalence")
    axes[1].set(title=f"Precision–Recall curve — {model_label}", xlabel="Recall", ylabel="Precision")
    axes[1].legend(loc="best")

    for axis in axes:
        axis.grid(alpha=0.25)
    fig.suptitle(f"{model_label} — chronological test holdout")
    fig.tight_layout()
    plt.show()


## 31. Feature Importance Selector

Change the model name in the comment below to inspect the fitted tree model. Importance is a screening aid; correlated features can share or distort importance.

In [ ]:
# Choose "Random Forest" or "XGBoost" for tree-based feature importance.
IMPORTANCE_MODEL_NAME = "Random Forest"
importance_model = tuned_models[IMPORTANCE_MODEL_NAME]
importance_estimator = importance_model.named_steps["model"]
importance_values = importance_estimator.feature_importances_
importance_names = importance_model.named_steps["preprocess"].get_feature_names_out()
importance_table = pd.DataFrame({"Feature": importance_names, "Importance": importance_values}).sort_values("Importance", ascending=False)
display(importance_table.head(20))


## 32. Permutation Importance

This shared code uses the same model selector and validation rows for all three tuned ensembles. It permutes raw input columns and scores the configured F1 criterion.

In [ ]:
perm_model_name = IMPORTANCE_MODEL_NAME
perm_model = tuned_models[perm_model_name]
X_perm = X_valid
perm = permutation_importance(perm_model, X_perm, y_valid, scoring=F1_SCORING, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
permutation_table = pd.DataFrame({"Feature": X_valid.columns, "Importance_Mean": perm.importances_mean, "Importance_Std": perm.importances_std}).sort_values("Importance_Mean", ascending=False)
display(permutation_table.head(20))


## 33. SHAP Selector

The model can be changed in one place. SHAP is optional; if it is installed, this cell tries a sample-based global explanation for the selected tuned model.

In [ ]:
# Choose "Random Forest" or "XGBoost" for optional tree SHAP plots.
SHAP_MODEL_NAME = "Random Forest"
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP is not installed. Optional install: %pip install shap")
if SHAP_AVAILABLE:
    shap_sample = X_valid.sample(n=min(300, len(X_valid)), random_state=RANDOM_STATE)
    shap_pipeline = tuned_models[SHAP_MODEL_NAME]
    shap_estimator = shap_pipeline.named_steps["model"]
    shap_input = shap_pipeline.named_steps["preprocess"].transform(shap_sample)
    try:
        shap_explainer = shap.TreeExplainer(shap_estimator)
        shap_values = shap_explainer(shap_input)
        if len(shap_values.values.shape) == 3: shap_values = shap_values[:, :, 1]
        shap.plots.bar(shap_values, max_display=20)
        shap.plots.beeswarm(shap_values, max_display=20)
    except Exception as error:
        print(f"SHAP could not explain {SHAP_MODEL_NAME}: {type(error).__name__}: {error}")
else:
    print("Skipped optional SHAP plots.")


## 34. Refit Models for Prediction

Models are refitted on all labelled rows. Section 35 lets you choose one tuned model or the final ensemble for prediction.

In [ ]:
if FINAL_ENSEMBLE_NAME not in ensemble_members:
    raise ValueError(f"Unknown final ensemble: {FINAL_ENSEMBLE_NAME!r}")
final_threshold = ensemble_thresholds[FINAL_ENSEMBLE_NAME]
final_member_names = ensemble_members[FINAL_ENSEMBLE_NAME]
print("Validation-selected final ensemble:", FINAL_ENSEMBLE_NAME)
print("Ensemble threshold (selected on validation):", final_threshold)

# Fit every tuned model so the prediction cell can use a simple model selector.
final_models = {}
for model_name in tuned_models:
    if model_name == "Logistic Regression":
        model = Pipeline([
            ("preprocess", clone(preprocess_standard)),
            ("model", LogisticRegression(**logistic_best_params, solver="liblinear", max_iter=1500, random_state=RANDOM_STATE)),
        ])
    elif model_name == "Random Forest":
        model = Pipeline([("preprocess", clone(preprocess_tree)), ("model", RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=RANDOM_STATE))])
    elif model_name == "XGBoost":
        model = Pipeline([("preprocess", clone(preprocess_tree)), ("model", XGBClassifier(**xgb_best_params, eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE))])
    fit_model(model_name, model, X, y)
    final_models[model_name] = model


## 35. Choose a Model and Predict on the Test Dataset

Select a tuned model or the validation-selected final ensemble below. The chosen validation threshold is applied to predictions.

In [ ]:
# Choose: "Final ensemble", "Logistic Regression", "Random Forest", or "XGBoost".
PREDICTION_MODEL_NAME = "Final ensemble"
PREDICTION_CHOICES = ["Final ensemble", *tuned_models.keys()]
if PREDICTION_MODEL_NAME not in PREDICTION_CHOICES:
    raise ValueError(f"Choose one of: {PREDICTION_CHOICES}")

if PREDICTION_MODEL_NAME == "Final ensemble":
    prediction_threshold = final_threshold
    prediction_members = final_member_names
    member_probabilities = [
        predict_probability(name, final_models[name], X_predict)
        for name in prediction_members
    ]
    predict_probability_final = np.mean(member_probabilities, axis=0)
else:
    prediction_threshold = thresholds[PREDICTION_MODEL_NAME]
    predict_probability_final = predict_probability(
        PREDICTION_MODEL_NAME, final_models[PREDICTION_MODEL_NAME], X_predict
    )

predict_label_final = (predict_probability_final >= prediction_threshold).astype(int)
print("Prediction model:", PREDICTION_MODEL_NAME)
print("Prediction threshold:", round(float(prediction_threshold), 4))
print("Prediction rows:", len(predict_label_final))
print("Predicted positive rate:", round(float(predict_label_final.mean()), 4))

# Keep every existing test column and append the two prediction columns.
test["predict_proba"] = predict_probability_final
test["predict"] = predict_label_final
display(test.head())


## 36. Final Run Summary

Use these outputs in the report: baseline metrics, RandomizedSearchCV best parameters and CV F1, validation thresholds, validation ensemble comparison, before/after refit test comparison, and labelled test metrics when available.

In [ ]:
run_summary = pd.DataFrame({
    "Item": ["F1 definition", "Positive label", "Final ensemble", "Locked validation threshold", "Labelled rows", "Unlabelled prediction rows", "Train rows", "Validation rows", "Test holdout rows", "Feature count"],
    "Value": [F1_AVERAGE, POSITIVE_LABEL, FINAL_ENSEMBLE_NAME, final_threshold, len(X), len(X_predict), len(X_train), len(X_valid), len(X_test_split), X.shape[1]],
})
display(run_summary)
